# Dtypes in PyTorch 

## What a dtype is

The data type of a tensor's elements — defines how much memory each element uses and how it's interpreted (float, integer, boolean, etc). Same concept as dtypes in NumPy.

```python
x = torch.rand(3)
x.dtype   # → torch.float32
```

---

## Default dtype: `torch.float32`

If you create a tensor from floating-point values without specifying a dtype, it defaults to `torch.float32`:

```python
torch.tensor([1.0, 2.0])   # → dtype: float32
torch.rand(3)               # → dtype: float32
torch.zeros(3)               # → dtype: float32
```

Integer values default to `torch.int64` instead:

```python
torch.tensor([1, 2])   # → dtype: int64
```

`float32` is the standard "full precision" type used for weights, activations, and most computation in deep learning — it's the safe default PyTorch falls back to.

---

## Main dtypes

**Floating point:**

| dtype | Bits | Typical use |
|---|---|---|
| `torch.float32` / `torch.float` | 32 | default, standard precision |
| `torch.float64` / `torch.double` | 64 | more precision, more memory — rarely needed in deep learning |
| `torch.float16` / `torch.half` | 16 | mixed precision training, faster + less memory |
| `torch.bfloat16` | 16 | like float16 but wider dynamic range — common on modern GPUs/TPUs |

**Integer:**

| dtype | Bits |
|---|---|
| `torch.int64` / `torch.long` | 64 |
| `torch.int32` / `torch.int` | 32 |
| `torch.int16`, `torch.int8` | 16, 8 |
| `torch.uint8` | 8 (unsigned) |

**Boolean:** `torch.bool`

---

## Setting the dtype at creation time

You can specify the dtype right when the tensor is created, instead of converting it afterward:

```python
x = torch.tensor([1.0, 2.0], dtype=torch.float64)
x = torch.zeros(3, dtype=torch.int64)
x = torch.rand(3, dtype=torch.float32)
```

This is generally preferable to creating with the default and converting later — one less operation, and no risk of forgetting to convert before it's used somewhere that expects a specific dtype.

---

## Changing dtype after creation — with `.to()`

If a tensor already exists with the wrong dtype, `.to()` converts it. Same method used for changing device, just with a dtype argument instead:

```python
x = torch.rand(3)               # float32
x = x.to(torch.float64)         # now float64
x = x.to(dtype=torch.int64)     # now int64 (explicit keyword form)
```

Like with device, `.to()` on a plain tensor is **not in-place** — it returns a new tensor, so you must reassign it:

```python
x.to(torch.float64)      # ❌ result discarded
x = x.to(torch.float64)  # ✅ correct
```

You can also change device and dtype in the same call:

```python
x = x.to(device="cuda", dtype=torch.float16)
```

### Shortcut methods (equivalent to `.to(dtype)`)

```python
x.float()     # → torch.float32
x.double()    # → torch.float64
x.half()      # → torch.float16
x.bfloat16()  # → torch.bfloat16
x.long()      # → torch.int64   (most common — used for classification labels)
x.int()       # → torch.int32
x.bool()      # → torch.bool
```

---

## Why it matters in practice

- **Mismatched dtypes between tensors** can raise errors or unexpected behavior, especially on GPU-specific kernels that expect exact dtype matches.
- **Classification labels usually need `int64`/`long`** — `nn.CrossEntropyLoss` expects targets as `long`, and a common beginner error is passing `float32` labels (e.g. loaded from a generic NumPy array) by mistake.
- **`float16`/`bfloat16` use half the memory of `float32`** and are typically faster on modern GPUs with Tensor Cores — this is the basis of mixed precision training.

---

## Summary

| Task | How |
|---|---|
| Check current dtype | `x.dtype` |
| Default for floats | `torch.float32` |
| Default for ints | `torch.int64` |
| Set at creation | `torch.tensor(data, dtype=torch.float64)` |
| Convert after creation | `x = x.to(torch.float64)` |
| Convert with a shortcut | `x = x.long()`, `x = x.float()`, etc. |

In [5]:
import torch

a = torch.tensor(2)
print(a.dtype)

torch.int64


In [6]:
a = a.to(torch.float16)
print(a.dtype)

torch.float16


In [10]:
import torch.nn as nn

b = nn.Linear(2, 2)
print(b.weight.dtype)

torch.float32
